In [2]:
import pandas as pd

In [24]:
import pandas as pd

df = pd.DataFrame({
    'poi_id': [101, 101, 102, 102, 102, 103],
    'omzet': [100, 150, 200, 50, 75, 300],
    'kosten': [40, 60, 120, 20, 30, 150],
    'aantal': [10, 15, 20, 5, 8, 25]
})

df['datum'] = pd.to_datetime([
    '2024-01-01',
    '2024-01-02',
    '2024-01-01',
    '2024-01-02',
    '2024-01-02',
    '2024-01-01'
])

df

,poi_id,omzet,kosten,aantal,datum
0,101,100,40,10,2024-01-01
1,101,150,60,15,2024-01-02
2,102,200,120,20,2024-01-01
3,102,50,20,5,2024-01-02
4,102,75,30,8,2024-01-02
5,103,300,150,25,2024-01-01


In [27]:
# df.groupby('poi_id', as_index=False)[['omzet', 'kosten', 'aantal']].sum()
df.groupby(['poi_id', 'datum'], as_index=False)[['omzet', 'kosten', 'aantal']].sum()

,poi_id,datum,omzet,kosten,aantal
0,101,2024-01-01,100,40,10
1,101,2024-01-02,150,60,15
2,102,2024-01-01,200,120,20
3,102,2024-01-02,125,50,13
4,103,2024-01-01,300,150,25


In [8]:
area_meta_df = pd.read_csv("area_meta_cleaned.csv")
poi_df = pd.read_csv("poi_matches_MLP_cleaned.csv")

In [9]:
poi_df.head()

,date,poi_id,matches,month,day_name,is_weekend,Seizoen,point
0,2025-01-02,18699,5,januari,Thursday,0,herfst,POINT(4.702411389059087 50.99397698611658)
1,2025-01-04,18699,5,januari,Saturday,1,herfst,POINT(4.702411389059087 50.99397698611658)
2,2025-01-05,18699,5,januari,Sunday,1,herfst,POINT(4.702411389059087 50.99397698611658)
3,2025-01-06,18699,4,januari,Monday,0,herfst,POINT(4.702411389059087 50.99397698611658)
4,2025-01-07,18699,1,januari,Tuesday,0,herfst,POINT(4.702411389059087 50.99397698611658)


In [10]:
area_meta_df.head()

,area_id,geometry,1 person,2 persons,3 persons,4 persons,5 persons,From 10 to 14 years (Total female population),From 10 to 14 years (Total male population),From 15 to 19 years (Total female population),...,Persons under the national minimum age for economic activity,No diploma or certificate,Primary education,Lower secondary education,Upper secondary education,Post-secondary non tertiary education,Short cycle Tertiary education,Bachelors or equivalent,Masters or equivalent,Doctorate or equivalent
0,1-6-82032B00-,"POLYGON((5.95935985763697 50.3094619520165, 5....",26.0,42.0,17.0,23.0,8.0,5.0,9.0,8.0,...,68.0,5.0,24.0,48.0,73.0,9.0,0.0,65.0,12.0,0.0
1,1-6-82037B19-,"POLYGON((6.02712762297411 50.2098971823183, 6....",9.0,8.0,2.0,2.0,1.0,1.0,0.0,1.0,...,4.0,0.0,6.0,9.0,12.0,1.0,0.0,5.0,0.0,0.0
2,1-6-82037D00-,"POLYGON((5.85889815596349 50.1770549579093, 5....",14.0,16.0,10.0,10.0,2.0,1.0,5.0,3.0,...,25.0,2.0,7.0,17.0,37.0,4.0,0.0,22.0,10.0,1.0
3,1-6-83012A001,"POLYGON((5.45578148067834 50.3503693193278, 5....",20.0,6.0,2.0,4.0,2.0,1.0,0.0,4.0,...,7.0,0.0,5.0,12.0,20.0,1.0,0.0,13.0,4.0,0.0
4,1-6-83012E01-,"POLYGON((5.49576698834463 50.3519634152142, 5....",50.0,32.0,7.0,6.0,1.0,3.0,5.0,1.0,...,22.0,4.0,22.0,38.0,58.0,3.0,0.0,21.0,6.0,1.0


In [11]:
poi_df.shape

(53894, 8)

dus in pois heb je een point.
in areas heb je een geom waarbinnen censusdata valt.
1) join areas met area_meta op basis van area_id om zo de kolom geom aan area_meta te geven
2) voor elke rij in in df_merged -> ga na welk punt in geom het dichtste bij point ligt van pois. Deze moet dichter liggen dan 5km van het punt. ga na welk punt in geom het verste van point ligt van pois. Deze moet dichter liggen dan 5.2km van het punt. alle rijen die hieraan voldoen worden bij elkaar opgetelt worden en bij die poi gezet.

In [37]:
import pandas as pd
import geopandas as gpd
from shapely import wkt

In [38]:
df_poi = pd.read_csv("poi_matches_MLP_cleaned.csv")
df_points = df_poi[["poi_id", "date", "point"]].copy()
df_points["point"] = df_points["point"].apply(wkt.loads)
gdf_points = gpd.GeoDataFrame(df_points, geometry="point", crs="EPSG:4326")

In [39]:
df_polygons = pd.read_csv("area_meta_cleaned.csv")
df_polygons["geometry"] = df_polygons["geometry"].apply(wkt.loads)
gdf_polygons = gpd.GeoDataFrame(df_polygons, geometry="geometry", crs="EPSG:4326")

In [40]:
gdf_points_proj = gdf_points.to_crs(epsg=3857)
gdf_polygons_proj = gdf_polygons.to_crs(epsg=3857)

result = gpd.sjoin(
    gdf_points_proj,
    gdf_polygons_proj,
    how="left",
    predicate="dwithin",
    distance=5000
)

In [41]:
# na dit blijf je normaal gezien enkel over met poi_id, date en alle censusdata
result = result.drop(columns=["point", "area_id", "index_right"])

# result["points"] = df_points["point"]
# result = pd.merge(result, df_polygons, on='area_id', how='left')

In [42]:
cols = list(result.columns)
cols.remove("poi_id")
cols.remove("date")

In [44]:
result = result.groupby(['poi_id', 'date'], as_index=False)[cols].sum()

In [45]:
result

,poi_id,date,1 person,2 persons,3 persons,4 persons,5 persons,From 10 to 14 years (Total female population),From 10 to 14 years (Total male population),From 15 to 19 years (Total female population),...,Persons under the national minimum age for economic activity,No diploma or certificate,Primary education,Lower secondary education,Upper secondary education,Post-secondary non tertiary education,Short cycle Tertiary education,Bachelors or equivalent,Masters or equivalent,Doctorate or equivalent
0,18699,2025-01-02,2896.0,4379.0,1830.0,1701.0,500.0,753.0,834.0,761.0,...,4034.0,318.0,1907.0,4467.0,7374.0,1073.0,518.0,4393.0,2804.0,215.0
1,18699,2025-01-04,2896.0,4379.0,1830.0,1701.0,500.0,753.0,834.0,761.0,...,4034.0,318.0,1907.0,4467.0,7374.0,1073.0,518.0,4393.0,2804.0,215.0
2,18699,2025-01-05,2896.0,4379.0,1830.0,1701.0,500.0,753.0,834.0,761.0,...,4034.0,318.0,1907.0,4467.0,7374.0,1073.0,518.0,4393.0,2804.0,215.0
3,18699,2025-01-06,2896.0,4379.0,1830.0,1701.0,500.0,753.0,834.0,761.0,...,4034.0,318.0,1907.0,4467.0,7374.0,1073.0,518.0,4393.0,2804.0,215.0
4,18699,2025-01-07,2896.0,4379.0,1830.0,1701.0,500.0,753.0,834.0,761.0,...,4034.0,318.0,1907.0,4467.0,7374.0,1073.0,518.0,4393.0,2804.0,215.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
53889,422371,2026-03-25,5457.0,6771.0,3005.0,2719.0,1010.0,1559.0,1566.0,1371.0,...,8642.0,655.0,4287.0,8515.0,12575.0,1727.0,825.0,5951.0,3167.0,158.0
53890,422371,2026-03-26,5457.0,6771.0,3005.0,2719.0,1010.0,1559.0,1566.0,1371.0,...,8642.0,655.0,4287.0,8515.0,12575.0,1727.0,825.0,5951.0,3167.0,158.0
53891,422371,2026-03-27,5457.0,6771.0,3005.0,2719.0,1010.0,1559.0,1566.0,1371.0,...,8642.0,655.0,4287.0,8515.0,12575.0,1727.0,825.0,5951.0,3167.0,158.0
53892,422371,2026-03-28,5457.0,6771.0,3005.0,2719.0,1010.0,1559.0,1566.0,1371.0,...,8642.0,655.0,4287.0,8515.0,12575.0,1727.0,825.0,5951.0,3167.0,158.0


In [49]:
result = pd.merge(result, df_poi, on=['poi_id', 'date'], how='inner')
result = result.drop(columns=["point"])

In [50]:
result[["date", "month", "day_name", "is_weekend", "Seizoen"]]

,date,month,day_name,is_weekend,Seizoen
0,2025-01-02,januari,Thursday,0,winter
1,2025-01-04,januari,Saturday,1,winter
2,2025-01-05,januari,Sunday,1,winter
3,2025-01-06,januari,Monday,0,winter
4,2025-01-07,januari,Tuesday,0,winter
...,...,...,...,...,...
53889,2026-03-25,maart,Wednesday,0,lente
53890,2026-03-26,maart,Thursday,0,lente
53891,2026-03-27,maart,Friday,0,lente
53892,2026-03-28,maart,Saturday,1,lente
